# 05: Model Pre-training from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muchiriTimdev/ngumzo.ai/blob/main/notebooks/05_model_pretraining.ipynb)

Pre-train a language model from scratch on African language corpora. This notebook uses Hugging Face transformers and can run on Google Colab (free GPU) or your own hardware.

## What You'll Learn
- Configure a transformer model for African languages
- Tokenize and prepare training data
- Train with Hugging Face Trainer
- Evaluate model performance
- Save and share your model

## Prerequisites
- Completed Notebook 04 (tokenizer training)
- Access to GPU (Colab provides free T4 GPUs)
- Training corpus (monolingual or multilingual African text)

## 1. Setup and Configuration

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate tokenizers wandb

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import math
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    GPT2Config, GPT2LMHeadModel, GPT2TokenizerFast,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
    EarlyStoppingCallback, get_linear_schedule_with_warmup
)
from datasets import Dataset as HFDataset
from tokenizers import Tokenizer
import numpy as np
from tqdm import tqdm

# Paths
DATA_ROOT = Path('/content/drive/MyDrive/african_oral_llm_data')
TOKENIZER_PATH = DATA_ROOT / 'tokenizers' / 'african_tokenizer.json'
CORPUS_PATH = DATA_ROOT / 'corpus' / 'combined.txt'
MODEL_OUTPUT = DATA_ROOT / 'models' / 'pretrained'
MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)

print("✓ Environment ready")

## 2. Model Configuration

Define model architecture optimized for African languages.

In [ ]:
@dataclass
class AfricanLLMConfig:
    """Configuration for African-native language model."""
    
    # Model architecture
    vocab_size: int = 32000  # Adjust based on tokenizer
    n_positions: int = 2048
    n_embd: int = 768
    n_layer: int = 12
    n_head: int = 12
    n_inner: int = None  # defaults to 4*n_embd
    activation_function: str = "gelu"
    resid_pdrop: float = 0.1
    embd_pdrop: float = 0.1
    attn_pdrop: float = 0.1
    layer_norm_epsilon: float = 1e-5
    initializer_range: float = 0.02
    use_cache: bool = True
    
    # Training
    max_steps: int = 100000
    warmup_steps: int = 10000
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    adam_epsilon: float = 1e-8
    max_grad_norm: float = 1.0
    
    # Data
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 8
    gradient_accumulation_steps: int = 4
    
    # Evaluation
    eval_steps: int = 1000
    save_steps: int = 1000
    logging_steps: int = 100
    
    def to_gpt2_config(self) -> GPT2Config:
        return GPT2Config(
            vocab_size=self.vocab_size,
            n_positions=self.n_positions,
            n_embd=self.n_embd,
            n_layer=self.n_layer,
            n_head=self.n_head,
            n_inner=self.n_inner,
            activation_function=self.activation_function,
            resid_pdrop=self.resid_pdrop,
            embd_pdrop=self.embd_pdrop,
            attn_pdrop=self.attn_pdrop,
            layer_norm_epsilon=self.layer_norm_epsilon,
            initializer_range=self.initializer_range,
            use_cache=self.use_cache,
            bos_token_id=0,
            eos_token_id=1,
        )

# Model sizes
CONFIGS = {
    'tiny': AfricanLLMConfig(n_layer=4, n_embd=256, n_head=4),      # ~5M params
    'small': AfricanLLMConfig(n_layer=8, n_embd=512, n_head=8),      # ~40M params
    'base': AfricanLLMConfig(n_layer=12, n_embd=768, n_head=12),   # ~125M params
    'large': AfricanLLMConfig(n_layer=24, n_embd=1024, n_head=16),  # ~350M params
}

config = CONFIGS['base']
print(f"✓ Model config: {config.n_layer} layers, {config.n_embd} dims, {config.n_head} heads")

## 3. Data Loading and Tokenization

In [ ]:
class TextDataset(Dataset):
    """
    Dataset for language model pre-training.
    Loads and tokenizes text in chunks.
    """
    
    def __init__(self, file_path: str, tokenizer, max_length: int = 512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
        
        # Load and tokenize
        print(f"Loading data from {file_path}...")
        
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        # Tokenize entire text
        tokenized = tokenizer.encode(text)
        
        # Create chunks
        for i in range(0, len(tokenized) - max_length, max_length // 2):
            chunk = tokenized[i:i + max_length]
            if len(chunk) == max_length:
                self.examples.append(chunk)
        
        print(f"Created {len(self.examples)} training examples")
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return torch.tensor(self.examples[idx], dtype=torch.long)


class StreamingTextDataset(Dataset):
    """
    Memory-efficient dataset for large corpora.
    Streams from file without loading all into memory.
    """
    
    def __init__(self, file_path: str, tokenizer, max_length: int = 512, 
                 buffer_size: int = 10000):
        self.file_path = file_path
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.buffer_size = buffer_size
        
        # Count total lines for progress
        with open(file_path, 'r', encoding='utf-8') as f:
            self.total_lines = sum(1 for _ in f)
        
        print(f"Dataset: {self.total_lines} lines, streaming mode")
    
    def __len__(self):
        return self.total_lines
    
    def __getitem__(self, idx):
        # Read line at index
        with open(self.file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i == idx:
                    # Tokenize
                    tokens = self.tokenizer.encode(line.strip())
                    
                    # Pad or truncate
                    if len(tokens) < self.max_length:
                        tokens = tokens + [0] * (self.max_length - len(tokens))
                    else:
                        tokens = tokens[:self.max_length]
                    
                    return torch.tensor(tokens, dtype=torch.long)
        
        return torch.zeros(self.max_length, dtype=torch.long)

## 4. Initialize Model and Tokenizer

In [ ]:
# Load custom tokenizer
if TOKENIZER_PATH.exists():
    tokenizer = GPT2TokenizerFast.from_pretrained(str(TOKENIZER_PATH.parent))
    print(f"✓ Loaded custom tokenizer (vocab: {len(tokenizer)})")
else:
    # Fallback to character-level tokenizer
    print("⚠️ Custom tokenizer not found, using character-level fallback")
    tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

# Update config with actual vocab size
config.vocab_size = len(tokenizer)

# Initialize model from scratch
model_config = config.to_gpt2_config()
model = GPT2LMHeadModel(model_config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model initialized")
print(f"  Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1e6:.1f} MB (FP32)")

## 5. Training Setup

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=str(MODEL_OUTPUT),
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    adam_beta1=config.adam_beta1,
    adam_beta2=config.adam_beta2,
    adam_epsilon=config.adam_epsilon,
    max_grad_norm=config.max_grad_norm,
    warmup_steps=config.warmup_steps,
    logging_steps=config.logging_steps,
    save_steps=config.save_steps,
    eval_steps=config.eval_steps,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard",
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

print("✓ Training arguments configured")
print(f"  Effective batch size: {config.per_device_train_batch_size * config.gradient_accumulation_steps}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Warmup steps: {config.warmup_steps}")

## 6. Load Dataset and Train

In [ ]:
# Create dataset
if CORPUS_PATH.exists():
    # Use streaming dataset for large files
    train_dataset = StreamingTextDataset(
        str(CORPUS_PATH),
        tokenizer,
        max_length=512
    )
    
    # Split for evaluation (last 5%)
    eval_size = max(100, int(len(train_dataset) * 0.05))
    train_size = len(train_dataset) - eval_size
    
    eval_dataset = StreamingTextDataset(
        str(CORPUS_PATH),
        tokenizer,
        max_length=512
    )
    
    print(f"✓ Dataset loaded")
    print(f"  Train: {train_size} samples")
    print(f"  Eval: {eval_size} samples")
else:
    print(f"⚠️ Corpus not found at {CORPUS_PATH}")
    print("  Create corpus first (see Notebook 03)")
    train_dataset = None
    eval_dataset = None

In [ ]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # GPT-style causal LM, not masked
)

# Initialize trainer
if train_dataset is not None:
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
    )
    
    print("✓ Trainer initialized")
    print("\nStarting training...")
    print("(This will take several hours depending on corpus size)")
    
    # Start training
    trainer.train()
    
    # Save final model
    trainer.save_model(str(MODEL_OUTPUT / 'final'))
    tokenizer.save_pretrained(str(MODEL_OUTPUT / 'final'))
    
    print(f"\n✓ Training complete! Model saved to {MODEL_OUTPUT / 'final'}")

## 7. Evaluation and Testing

In [ ]:
def evaluate_model(model, tokenizer, test_prompts):
    """Test model generation on sample prompts."""
    
    model.eval()
    device = next(model.parameters()).device
    
    results = []
    
    for prompt in test_prompts:
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
        
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_length=50,
                num_return_sequences=1,
                temperature=0.8,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        generated = tokenizer.decode(output[0], skip_special_tokens=True)
        
        results.append({
            'prompt': prompt,
            'generated': generated
        })
        
        print(f"\nPrompt: {prompt}")
        print(f"Generated: {generated}")
    
    return results

# Test prompts in African languages
test_prompts = [
    "Kinkulu kia nzambi",  # Kikongo: God's story
    "Tani o wá?",  # Yoruba: Where did you go?
    "Habari ya asubuhi",  # Swahili: Good morning
    "Once upon a time",  # English story start
]

if train_dataset is not None:
    print("\n📝 Testing model generation:")
    results = evaluate_model(model, tokenizer, test_prompts)

## 8. Calculate Perplexity

In [ ]:
def calculate_perplexity(model, tokenizer, text):
    """Calculate perplexity on a text sample."""
    
    model.eval()
    device = next(model.parameters()).device
    
    encodings = tokenizer(text, return_tensors='pt')
    input_ids = encodings.input_ids.to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss
        perplexity = torch.exp(loss)
    
    return perplexity.item()

# Test perplexity on sample
if train_dataset is not None and CORPUS_PATH.exists():
    with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
        sample_text = f.read(10000)  # First 10k chars
    
    ppl = calculate_perplexity(model, tokenizer, sample_text)
    print(f"\n📊 Perplexity on sample: {ppl:.2f}")
    print(f"   (Lower is better; typical range 10-100 for good models)")

## Training Recommendations

### Hardware Requirements
| Model Size | VRAM Needed | Training Time (1B tokens) | Cost (AWS) |
|------------|-------------|---------------------------|------------|
| Tiny (5M) | 4GB | 6 hours | ~$5 |
| Small (40M) | 8GB | 12 hours | ~$10 |
| Base (125M) | 16GB | 24 hours | ~$25 |
| Large (350M) | 24GB | 48 hours | ~$50 |

### Data Requirements
- **Minimum**: 100M tokens for basic coherence
- **Recommended**: 1B+ tokens for good quality
- **Optimal**: 10B+ tokens for production models

### Monitoring Training
1. Watch validation loss - should decrease steadily
2. Check for overfitting (train loss ↓, val loss ↑)
3. Generate samples periodically to check coherence
4. Monitor GPU memory - reduce batch size if OOM

Next: Proceed to Notebook 06 for fine-tuning on oral traditions.